# Activation Patching Tryout

In this notebook I will play around with activation patching to get a better understanding for it

Same setup as in the explore notebook: a random sequence repeated twice. New is the corrupted twin, where the first half is a different random sequence, so there is nothing to copy from. Both runs share the second half, only the first half differs.

In [1]:
# Setup: same GPT-2 small and repeated random sequence as in 01_explore, plus a corrupted twin.
# clean   = [BOS, A, A]  the second half can be copied from the first
# corrupt = [BOS, B, A]  same second half, but nothing to copy from
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2", device="mps")
torch.manual_seed(0)
L = 20
A = torch.randint(1000, 10000, (1, L))  # same seed and call order as 01_explore, so A equals rand there
B = torch.randint(1000, 10000, (1, L))
bos = torch.tensor([[model.tokenizer.bos_token_id]])
clean_tokens = torch.cat([bos, A, A], dim=1).to("mps")
corrupt_tokens = torch.cat([bos, B, A], dim=1).to("mps")
second_half = slice(L, None)  # positions L..2L make the predictions for the second half

/Users/Brose/dev/private_repos/mechinterp_xai/.venv/lib/python3.13/site-packages/transformer_lens/config/hooked_transformer_config.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.13.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


Baselines first. The gap between clean and corrupt is what we try to locate in the model. Restoration maps a patched loss onto that gap: 0 means nothing happened, 1 means fully back to clean.

In [2]:
# Baselines: second-half loss for both runs. Expectation: clean ~1.10 exactly as in 01_explore
# (same sequence), corrupt near the level of unrepeated random text (~12).
def second_half_loss(tokens, fwd_hooks=()):
    loss = model.run_with_hooks(tokens, return_type="loss", loss_per_token=True,
                                fwd_hooks=list(fwd_hooks))[0]
    return loss[second_half].mean().item()

clean_loss = second_half_loss(clean_tokens)
corrupt_loss = second_half_loss(corrupt_tokens)
print(f"clean {clean_loss:.2f}   corrupt {corrupt_loss:.2f}")

_, clean_cache = model.run_with_cache(clean_tokens)

def restoration(patched_loss):
    # 0 = the patch changed nothing, 1 = clean behaviour fully restored
    return (corrupt_loss - patched_loss) / (corrupt_loss - clean_loss)


clean 1.10   corrupt 12.01


Layer scan: swap in the whole residual stream of the second half, one layer at a time. This tells when the copied information arrives in the second half, not which head writes it.

In [3]:
# Layer scan: transplant the whole residual stream at the second-half positions, one layer at a time.
# Expectation: near 0 before the induction layers, near 1 after them, a step around layers 5-8,
# because the copied information only exists in the second half's residual stream once those
# heads have written it. Not exactly 1 before the very end: later layers still attend to the
# corrupted first half. The final resid_post must give exactly 1.00 by construction.
def patch_positions(clean_act):
    def hook(act, hook):
        act[:, second_half] = clean_act[:, second_half]
        return act
    return hook

names = [f"blocks.{l}.hook_resid_pre" for l in range(model.cfg.n_layers)] + ["blocks.11.hook_resid_post"]
for name in names:
    loss = second_half_loss(corrupt_tokens, [(name, patch_positions(clean_cache[name]))])
    print(f"{name:26s} restoration {restoration(loss):5.2f}")

blocks.0.hook_resid_pre    restoration  0.03
blocks.1.hook_resid_pre    restoration  0.04
blocks.2.hook_resid_pre    restoration  0.04
blocks.3.hook_resid_pre    restoration  0.09
blocks.4.hook_resid_pre    restoration  0.10
blocks.5.hook_resid_pre    restoration  0.10
blocks.6.hook_resid_pre    restoration  0.28
blocks.7.hook_resid_pre    restoration  0.46
blocks.8.hook_resid_pre    restoration  0.70
blocks.9.hook_resid_pre    restoration  0.75
blocks.10.hook_resid_pre   restoration  0.93
blocks.11.hook_resid_pre   restoration  0.99
blocks.11.hook_resid_post  restoration  1.00


Expected a step between layer 5 and 8. Got the step (0.28 / 0.46 / 0.70 after the three induction layers), but also a ramp up to layer 11. So 30% of the gap closes later than I thought.

Head scan: same thing, but only one head's output at a time. 144 forward passes.

In [ ]:
# Head scan: transplant one head's output (hook_z) at the second-half positions, all 144 heads.
# Expectation: the five induction heads from 01_explore top the list, each restoring only part of
# the gap on its own. If heads in layers 8-10 show up as well, the ramp above came from copying there.
def patch_head(clean_z, head):
    def hook(z, hook):  # z: [batch, pos, head, d_head]
        z[:, second_half, head, :] = clean_z[:, second_half, head, :]
        return z
    return hook

head_rest = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for l in range(model.cfg.n_layers):
    name = f"blocks.{l}.attn.hook_z"
    for h in range(model.cfg.n_heads):
        loss = second_half_loss(corrupt_tokens, [(name, patch_head(clean_cache[name], h))])
        head_rest[l, h] = restoration(loss)

torch.set_printoptions(precision=2, sci_mode=False, linewidth=140)
print(head_rest)
top = [(l, h, round(head_rest[l, h].item(), 2)) for l, h in zip(*torch.where(head_rest > 0.05))]
print(sorted(top, key=lambda t: -t[2])[:10])

tensor([[ 0.00, -0.03, -0.00, -0.01, -0.00, -0.00, -0.00, -0.00, -0.00, -0.00, -0.01, -0.00],
        [-0.00, -0.00,  0.00,  0.00, -0.00, -0.00, -0.00, -0.00,  0.00, -0.00, -0.00, -0.01],
        [ 0.00, -0.01, -0.01,  0.00, -0.00, -0.00,  0.00, -0.00,  0.00, -0.00, -0.00,  0.00],
        [-0.01,  0.01,  0.00, -0.01,  0.00, -0.01,  0.01,  0.00,  0.00, -0.00,  0.01,  0.00],
        [ 0.00,  0.00,  0.00, -0.00,  0.01,  0.00, -0.01,  0.01,  0.01,  0.00,  0.00,  0.00],
        [ 0.04,  0.08, -0.00,  0.00,  0.00,  0.05,  0.00, -0.00, -0.00, -0.01,  0.00, -0.01],
        [ 0.00,  0.00, -0.00, -0.00,  0.01,  0.00, -0.01,  0.00,  0.00,  0.12,  0.01,  0.00],
        [ 0.01,  0.02,  0.13, -0.00,  0.00,  0.00, -0.03,  0.03,  0.00, -0.00,  0.09,  0.01],
        [-0.00,  0.05, -0.00,  0.01,  0.00,  0.00,  0.04,  0.01,  0.01, -0.00, -0.04,  0.01],
        [ 0.02,  0.02,  0.04,  0.02,  0.00, -0.02,  0.11, -0.01,  0.01,  0.11,  0.00,  0.03],
        [ 0.06,  0.07,  0.02,  0.03,  0.02,  0.00,  0.09, -0

No single head does much, max 0.13. The ranking also does not match the induction scores from the explore notebook: L5H5 has the cleanest attention pattern but almost no causal effect here, L7H2 the other way round. Heads from layers 9 to 11 show up as well, and two heads (L10H7, L11H10) even hurt when patched in. No idea why yet.

Patching sets of heads together and comparing with the sum of the single effects. Random heads from the same layers as control, the same five as in the ablation.

In [ ]:
# Joint patches: sets of heads together, compared with the sum of their single effects.
# Expectation: the five induction heads together restore more than their sum (0.47), because the
# readout needs enough combined signal; five random heads from the same layers restore ~0; adding
# the late heads from layers 9-11 closes part of the remaining gap.
induction_heads = [(5, 1), (5, 5), (6, 9), (7, 2), (7, 10)]
late_heads = [(9, 6), (9, 9), (10, 6), (11, 9)]
torch.manual_seed(1)
pool = [(l, h) for l in (5, 6, 7) for h in range(model.cfg.n_heads) if (l, h) not in induction_heads]
random_heads = [pool[i] for i in torch.randperm(len(pool))[:5]]

def joint_restoration(heads):
    hooks = [(f"blocks.{l}.attn.hook_z", patch_head(clean_cache[f"blocks.{l}.attn.hook_z"], h))
             for l, h in heads]
    return restoration(second_half_loss(corrupt_tokens, hooks))

for label, heads in [("induction", induction_heads), ("random", random_heads),
                     ("induction + late", induction_heads + late_heads)]:
    single_sum = sum(head_rest[l, h].item() for l, h in heads)
    print(f"{label:17s} joint {joint_restoration(heads):5.2f}   sum of singles {single_sum:5.2f}   {heads}")

induction         joint  0.60   sum of singles  0.47   [(5, 1), (5, 5), (6, 9), (7, 2), (7, 10)]
random            joint  0.05   sum of singles  0.05   [(6, 0), (7, 11), (6, 4), (7, 7), (7, 9)]
induction + late  joint  0.86   sum of singles  0.86   [(5, 1), (5, 5), (6, 9), (7, 2), (7, 10), (9, 6), (9, 9), (10, 6), (11, 9)]


## What I take from this

The five induction heads together restore 0.60 of the gap, more than the sum of their single effects (0.47). Adding four late heads gets to 0.86, random heads do nothing (0.05). Removing the same five heads in the explore notebook only destroyed 25% of the gap, so they are more sufficient than necessary: when they are gone, other heads step in.

Ablation measures what is necessary, patching what is sufficient, and you need both to see that the mechanism is spread over many heads. Attention patterns tell where a head looks, not how much its output matters.

All of this is one sequence of 20 random tokens, so orientation, not statistics. Open question: loss vs logit difference as the metric.